In [0]:
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor

import warnings

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
)


INPUT_FILE = "Critical Parts AI Training.csv"
OUTPUT_FOLDER = Path("forecast_results")

# Rolling backtest settings
VALIDATION_START = "2026-01-01"
FORECAST_MONTHS = 6

TARGET_COLUMN = "Monthly Inventory Issues"
PART_COLUMN = "fpartno"
DATE_COLUMN = "Date"

# Commitment feature
COMMITS_COLUMN = "Avg Commits/Month"

LAGS = [1, 2, 3, 6, 12]

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "Version 8 usage + commitments + "
    "recency weighting settings loaded."
)

Version 8 usage + commitments + recency weighting settings loaded.


In [0]:
# =====================================================
# FILTER PARTS WITH AT LEAST 18 NONZERO USAGE MONTHS
# =====================================================

MIN_NONZERO_MONTHS = 18

validation_start = pd.Timestamp(
    VALIDATION_START
)

history_before_validation = data[
    data[DATE_COLUMN] < validation_start
].copy()


# Count months where actual inventory issues were > 0
nonzero_history = (
    history_before_validation[
        history_before_validation[
            TARGET_COLUMN
        ] > 0
    ]
    .groupby(PART_COLUMN)[DATE_COLUMN]
    .nunique()
    .reset_index(
        name="Nonzero Usage Months"
    )
)


# Make sure parts with zero nonzero months
# are still represented in the check
all_parts = (
    history_before_validation[
        [PART_COLUMN]
    ]
    .drop_duplicates()
)


part_history_check = (
    all_parts
    .merge(
        nonzero_history,
        on=PART_COLUMN,
        how="left",
    )
)


part_history_check[
    "Nonzero Usage Months"
] = (
    part_history_check[
        "Nonzero Usage Months"
    ]
    .fillna(0)
    .astype(int)
)


# =====================================================
# ELIGIBLE / EXCLUDED PARTS
# =====================================================

eligible_parts = (
    part_history_check[
        part_history_check[
            "Nonzero Usage Months"
        ] >= MIN_NONZERO_MONTHS
    ][PART_COLUMN]
)


excluded_parts = (
    part_history_check[
        part_history_check[
            "Nonzero Usage Months"
        ] < MIN_NONZERO_MONTHS
    ]
    .copy()
)


# =====================================================
# FILTER MAIN DATASET
# =====================================================

original_part_count = (
    data[PART_COLUMN].nunique()
)


data = data[
    data[PART_COLUMN].isin(
        eligible_parts
    )
].copy()


print(
    f"Minimum nonzero usage months required: "
    f"{MIN_NONZERO_MONTHS}"
)

print(
    f"Original parts: "
    f"{original_part_count:,}"
)

print(
    f"Parts included: "
    f"{data[PART_COLUMN].nunique():,}"
)

print(
    f"Parts excluded: "
    f"{len(excluded_parts):,}"
)


if not excluded_parts.empty:

    print("\nExcluded parts:")

    display(
        excluded_parts
        .sort_values(
            "Nonzero Usage Months"
        )
        .reset_index(drop=True)
    )

Minimum nonzero usage months required: 18
Original parts: 319
Parts included: 319
Parts excluded: 0


In [0]:
data = pd.read_csv(INPUT_FILE)

required_columns = {
    PART_COLUMN,
    DATE_COLUMN,
    TARGET_COLUMN,
    COMMITS_COLUMN,
}

missing_columns = required_columns.difference(
    data.columns
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: "
        f"{sorted(missing_columns)}"
    )

data[DATE_COLUMN] = pd.to_datetime(
    data[DATE_COLUMN],
    errors="raise",
)

# Keep only data from 2023 onward
data = data[
    data[DATE_COLUMN] >= pd.Timestamp("2023-01-01")
].copy()

data[TARGET_COLUMN] = pd.to_numeric(
    data[TARGET_COLUMN],
    errors="raise",
)

data[COMMITS_COLUMN] = pd.to_numeric(
    data[COMMITS_COLUMN],
    errors="coerce",
)

# Blank commitments treated as zero
data[COMMITS_COLUMN] = (
    data[COMMITS_COLUMN]
    .fillna(0)
)

data = (
    data
    .dropna(
        subset=[
            PART_COLUMN,
            DATE_COLUMN,
            TARGET_COLUMN,
        ]
    )
    .sort_values(
        [
            PART_COLUMN,
            DATE_COLUMN,
        ]
    )
    .reset_index(drop=True)
)

duplicates = data.duplicated(
    [
        PART_COLUMN,
        DATE_COLUMN,
    ],
    keep=False,
)

if duplicates.any():

    duplicate_rows = data.loc[
        duplicates,
        [
            PART_COLUMN,
            DATE_COLUMN,
        ],
    ]

    raise ValueError(
        "Duplicate part/month rows were found:\n"
        f"{duplicate_rows.head(20)}"
    )

print(f"Rows loaded: {len(data):,}")
print(f"Unique parts: {data[PART_COLUMN].nunique():,}")
print(f"First date: {data[DATE_COLUMN].min()}")
print(f"Last date: {data[DATE_COLUMN].max()}")
print(f"Average commitments: {data[COMMITS_COLUMN].mean():.2f}")
print(
    f"Rows with commitments: "
    f"{(data[COMMITS_COLUMN] > 0).sum():,}"
)

Rows loaded: 19,545
Unique parts: 449
First date: 2023-01-01 00:00:00
Last date: 2026-08-01 00:00:00
Average commitments: 4496.35
Rows with commitments: 14,716


In [0]:
def create_training_features(historical_data):

    feature_data = historical_data.copy()

    feature_data["month"] = feature_data[DATE_COLUMN].dt.month
    feature_data["year"] = feature_data[DATE_COLUMN].dt.year
    feature_data["quarter"] = feature_data[DATE_COLUMN].dt.quarter
    feature_data["time_idx"] = np.arange(len(feature_data))

    for lag in LAGS:
        feature_data[f"lag_{lag}"] = (
            feature_data[TARGET_COLUMN].shift(lag)
        )

    prior_usage = feature_data[TARGET_COLUMN].shift(1)

    feature_data["rolling_mean_3"] = prior_usage.rolling(3).mean()
    feature_data["rolling_mean_6"] = prior_usage.rolling(6).mean()
    feature_data["rolling_mean_12"] = prior_usage.rolling(12).mean()

    feature_data["rolling_median_3"] = prior_usage.rolling(3).median()
    feature_data["rolling_median_6"] = prior_usage.rolling(6).median()
    feature_data["rolling_median_12"] = prior_usage.rolling(12).median()

    feature_data["rolling_total_12"] = prior_usage.rolling(12).sum()

    feature_data["rolling_std_3"] = prior_usage.rolling(3).std()
    feature_data["rolling_std_6"] = prior_usage.rolling(6).std()
    feature_data["rolling_std_12"] = prior_usage.rolling(12).std()

    feature_data["rolling_min_12"] = prior_usage.rolling(12).min()
    feature_data["rolling_max_12"] = prior_usage.rolling(12).max()

    feature_data["trend_3"] = (
        feature_data["lag_1"] - feature_data["lag_3"]
    )

    feature_data["trend_6"] = (
        feature_data["lag_1"] - feature_data["lag_6"]
    )

    feature_data["zero_month_percentage_12"] = (
        prior_usage
        .rolling(12)
        .apply(
            lambda values: (values == 0).mean(),
            raw=True,
        )
    )

    feature_data["coefficient_variation_12"] = (
        feature_data["rolling_std_12"]
        /
        feature_data["rolling_mean_12"].replace(
            0,
            np.nan,
        )
    )

    feature_data["recent_vs_annual"] = (
        feature_data["rolling_mean_3"]
        -
        feature_data["rolling_mean_12"]
    )

    prior_commits = feature_data[COMMITS_COLUMN].shift(1)

    feature_data["commits_lag_1"] = (
        feature_data[COMMITS_COLUMN].shift(1)
    )

    feature_data["commits_lag_2"] = (
        feature_data[COMMITS_COLUMN].shift(2)
    )

    feature_data["commits_lag_3"] = (
        feature_data[COMMITS_COLUMN].shift(3)
    )

    feature_data["commits_lag_6"] = (
        feature_data[COMMITS_COLUMN].shift(6)
    )

    feature_data["commits_rolling_mean_3"] = (
        prior_commits.rolling(3).mean()
    )

    feature_data["commits_rolling_mean_6"] = (
        prior_commits.rolling(6).mean()
    )

    feature_data["commits_rolling_mean_12"] = (
        prior_commits.rolling(12).mean()
    )

    feature_data["commits_rolling_sum_3"] = (
        prior_commits.rolling(3).sum()
    )

    feature_data["commits_rolling_sum_6"] = (
        prior_commits.rolling(6).sum()
    )

    feature_data["commits_recent_vs_annual"] = (
        feature_data["commits_rolling_mean_3"]
        -
        feature_data["commits_rolling_mean_12"]
    )

    feature_columns = [
        "month",
        "year",
        "quarter",
        "time_idx",

        "lag_1",
        "lag_2",
        "lag_3",
        "lag_6",
        "lag_12",

        "rolling_mean_3",
        "rolling_mean_6",
        "rolling_mean_12",

        "rolling_median_3",
        "rolling_median_6",
        "rolling_median_12",

        "rolling_total_12",

        "rolling_std_3",
        "rolling_std_6",
        "rolling_std_12",

        "rolling_min_12",
        "rolling_max_12",

        "zero_month_percentage_12",
        "coefficient_variation_12",
        "recent_vs_annual",

        "trend_3",
        "trend_6",

        "commits_lag_1",
        "commits_lag_2",
        "commits_lag_3",
        "commits_lag_6",

        "commits_rolling_mean_3",
        "commits_rolling_mean_6",
        "commits_rolling_mean_12",

        "commits_rolling_sum_3",
        "commits_rolling_sum_6",

        "commits_recent_vs_annual",
    ]

    training_rows = (
        feature_data
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna(
            subset=feature_columns
        )
        .reset_index(drop=True)
    )

    return (
        training_rows,
        feature_columns,
    )


print(
    "Version 8 usage + commitment "
    "feature function created."
)

Version 8 usage + commitment feature function created.


In [0]:
def train_model(training_rows, feature_columns):

    # Recency weighting:
    # oldest usable training row = 1.0
    # newest usable training row = 2.0
    sample_weights = np.linspace(
        1.0,
        2.0,
        len(training_rows),
    )

    model = LGBMRegressor(
        objective="poisson",
        n_estimators=250,
        learning_rate=0.05,
        max_depth=5,
        min_child_samples=10,
        reg_lambda=1.0,
        random_state=42,
        verbose=-1,
    )

    model.fit(
        training_rows[feature_columns],
        training_rows[TARGET_COLUMN],
        sample_weight=sample_weights,
    )

    return model


print(
    "Version 8 training function created "
    "with recency weighting."
)


Version 8 training function created with recency weighting.


In [0]:
def forecast_future_months(
    model,
    historical_data,
    feature_columns,
    forecast_months,
    part_number,
):

    forecast_history = (
        historical_data[
            [
                DATE_COLUMN,
                TARGET_COLUMN,
                COMMITS_COLUMN,
            ]
        ]
        .copy()
        .sort_values(DATE_COLUMN)
        .reset_index(drop=True)
    )

    predictions = []

    for _ in range(forecast_months):

        next_date = (
            forecast_history[DATE_COLUMN].max()
            + pd.DateOffset(months=1)
        )

        recent_usage = forecast_history[TARGET_COLUMN]
        recent_commits = forecast_history[COMMITS_COLUMN]

        future_row = {
            "month": next_date.month,
            "year": next_date.year,
            "quarter": next_date.quarter,
            "time_idx": len(forecast_history),
        }

        for lag in LAGS:
            future_row[f"lag_{lag}"] = (
                recent_usage.iloc[-lag]
            )

        last_3 = recent_usage.iloc[-3:]
        last_6 = recent_usage.iloc[-6:]
        last_12 = recent_usage.iloc[-12:]

        future_row["rolling_mean_3"] = last_3.mean()
        future_row["rolling_mean_6"] = last_6.mean()
        future_row["rolling_mean_12"] = last_12.mean()

        future_row["rolling_median_3"] = last_3.median()
        future_row["rolling_median_6"] = last_6.median()
        future_row["rolling_median_12"] = last_12.median()

        future_row["rolling_total_12"] = last_12.sum()

        future_row["rolling_std_3"] = last_3.std()
        future_row["rolling_std_6"] = last_6.std()
        future_row["rolling_std_12"] = last_12.std()

        future_row["rolling_min_12"] = last_12.min()
        future_row["rolling_max_12"] = last_12.max()

        future_row["zero_month_percentage_12"] = (
            last_12.eq(0).mean()
        )

        rolling_mean_12 = future_row["rolling_mean_12"]
        rolling_std_12 = future_row["rolling_std_12"]

        if rolling_mean_12 != 0:
            future_row["coefficient_variation_12"] = (
                rolling_std_12 / rolling_mean_12
            )
        else:
            future_row["coefficient_variation_12"] = 0.0

        future_row["recent_vs_annual"] = (
            future_row["rolling_mean_3"]
            -
            future_row["rolling_mean_12"]
        )

        future_row["trend_3"] = (
            recent_usage.iloc[-1]
            -
            recent_usage.iloc[-3]
        )

        future_row["trend_6"] = (
            recent_usage.iloc[-1]
            -
            recent_usage.iloc[-6]
        )

        last_commits_3 = recent_commits.iloc[-3:]
        last_commits_6 = recent_commits.iloc[-6:]
        last_commits_12 = recent_commits.iloc[-12:]

        future_row["commits_lag_1"] = recent_commits.iloc[-1]
        future_row["commits_lag_2"] = recent_commits.iloc[-2]
        future_row["commits_lag_3"] = recent_commits.iloc[-3]
        future_row["commits_lag_6"] = recent_commits.iloc[-6]

        future_row["commits_rolling_mean_3"] = (
            last_commits_3.mean()
        )

        future_row["commits_rolling_mean_6"] = (
            last_commits_6.mean()
        )

        future_row["commits_rolling_mean_12"] = (
            last_commits_12.mean()
        )

        future_row["commits_rolling_sum_3"] = (
            last_commits_3.sum()
        )

        future_row["commits_rolling_sum_6"] = (
            last_commits_6.sum()
        )

        future_row["commits_recent_vs_annual"] = (
            future_row["commits_rolling_mean_3"]
            -
            future_row["commits_rolling_mean_12"]
        )

        future_features = pd.DataFrame(
            [future_row],
            columns=feature_columns,
        )

        predicted_usage = float(
            model.predict(
                future_features
            )[0]
        )

        predicted_usage = max(
            0.0,
            predicted_usage,
        )

        predictions.append(
            {
                PART_COLUMN: part_number,
                DATE_COLUMN: next_date,
                "Predicted Usage": predicted_usage,
            }
        )

        new_history_row = pd.DataFrame(
            {
                DATE_COLUMN: [next_date],
                TARGET_COLUMN: [predicted_usage],
                COMMITS_COLUMN: [0],
            }
        )

        forecast_history = pd.concat(
            [
                forecast_history,
                new_history_row,
            ],
            ignore_index=True,
        )

    return pd.DataFrame(predictions)


print(
    "Version 8 commitment forecast "
    "function created."
)


Version 8 commitment forecast function created.


In [0]:
all_results = []
errors = []

validation_start = pd.Timestamp(
    VALIDATION_START
)

validation_months = FORECAST_MONTHS

part_numbers = sorted(
    data[PART_COLUMN]
    .dropna()
    .unique()
)

total_parts = len(part_numbers)


for part_index, part_number in enumerate(
    part_numbers,
    start=1,
):

    try:

        part_data = (
            data[
                data[PART_COLUMN]
                == part_number
            ]
            .copy()
            .sort_values(DATE_COLUMN)
            .reset_index(drop=True)
        )


        part_results = []


        for month_number in range(
            validation_months
        ):

            prediction_date = (
                validation_start
                +
                pd.DateOffset(
                    months=month_number
                )
            )


            historical_data = (
                part_data[
                    part_data[DATE_COLUMN]
                    < prediction_date
                ]
                .copy()
            )


            actual_row = (
                part_data[
                    part_data[DATE_COLUMN]
                    == prediction_date
                ][
                    [
                        DATE_COLUMN,
                        TARGET_COLUMN,
                    ]
                ]
                .copy()
            )


            if actual_row.empty:

                raise ValueError(
                    f"No actual usage found for "
                    f"{prediction_date:%Y-%m}"
                )


            training_rows, feature_columns = (
                create_training_features(
                    historical_data
                )
            )


            if training_rows.empty:

                raise ValueError(
                    f"No usable training rows for "
                    f"{prediction_date:%Y-%m}"
                )


            model = train_model(
                training_rows,
                feature_columns,
            )


            one_month_forecast = (
                forecast_future_months(
                    model=model,
                    historical_data=historical_data,
                    feature_columns=feature_columns,
                    forecast_months=1,
                    part_number=part_number,
                )
            )


            predicted_usage = (
                one_month_forecast[
                    "Predicted Usage"
                ].iloc[0]
            )


            actual_usage = (
                actual_row[
                    TARGET_COLUMN
                ].iloc[0]
            )


            error = (
                predicted_usage
                -
                actual_usage
            )


            part_results.append(
                {
                    PART_COLUMN: part_number,

                    DATE_COLUMN: (
                        prediction_date
                    ),

                    "Predicted Usage": (
                        predicted_usage
                    ),

                    "Actual Usage": (
                        actual_usage
                    ),

                    "Error": (
                        error
                    ),

                    "Absolute Error": (
                        abs(error)
                    ),

                    "Training Through": (
                        historical_data[
                            DATE_COLUMN
                        ].max()
                    ),
                }
            )


        all_results.append(
            pd.DataFrame(
                part_results
            )
        )


    except Exception as error:

        errors.append(
            {
                PART_COLUMN: (
                    str(part_number)
                ),

                "Error": (
                    str(error)
                ),
            }
        )


    # Only print progress every 25 parts
    if (
        part_index % 25 == 0
        or part_index == total_parts
    ):

        print(
            f"Processed "
            f"{part_index:,} of "
            f"{total_parts:,} parts"
        )


if not all_results:

    raise RuntimeError(
        "No rolling forecasts "
        "completed successfully."
    )


print(
    "\nVersion 8 rolling backtest complete."
)

print(
    f"Successful parts: "
    f"{len(all_results):,}"
)

print(
    f"Skipped parts: "
    f"{len(errors):,}"
)

Processed 25 of 449 parts


[LightGBM] [Fatal] [poisson]: sum of labels is zero


Processed 50 of 449 parts


[LightGBM] [Fatal] [poisson]: sum of labels is zero


[LightGBM] [Fatal] [poisson]: sum of labels is zero


...

Processed 400 of 449 parts


[LightGBM] [Fatal] [poisson]: sum of labels is zero


Processed 425 of 449 parts


[LightGBM] [Fatal] [poisson]: sum of labels is zero


Processed 449 of 449 parts

Version 8 rolling backtest complete.
Successful parts: 372
Skipped parts: 77


In [0]:
results = (
    pd.concat(
        all_results,
        ignore_index=True,
    )
    .sort_values(
        [
            PART_COLUMN,
            DATE_COLUMN,
        ]
    )
    .reset_index(drop=True)
)

results["Predicted Usage Rounded"] = (
    results["Predicted Usage"]
    .round()
    .astype(int)
)

results["Actual Usage Rounded"] = (
    results["Actual Usage"]
    .round()
    .astype(int)
)

display(
    results[
        [
            PART_COLUMN,
            DATE_COLUMN,
            "Training Through",
            "Predicted Usage Rounded",
            "Actual Usage Rounded",
            "Error",
            "Absolute Error",
        ]
    ]
)


,fpartno,Date,Training Through,Predicted Usage Rounded,Actual Usage Rounded,Error,Absolute Error
0,058825A,2026-01-01,2025-12-01,529,592,-62.559207,62.559207
1,058825A,2026-02-01,2026-01-01,600,989,-388.706107,388.706107
2,058825A,2026-03-01,2026-02-01,724,710,14.277097,14.277097
3,058825A,2026-04-01,2026-03-01,748,979,-230.910854,230.910854
4,058825A,2026-05-01,2026-04-01,866,1147,-281.010893,281.010893
...,...,...,...,...,...,...,...
2227,WAS-0175,2026-02-01,2026-01-01,9,10,-0.671924,0.671924
2228,WAS-0175,2026-03-01,2026-02-01,10,3,6.640887,6.640887
2229,WAS-0175,2026-04-01,2026-03-01,4,2,2.451332,2.451332
2230,WAS-0175,2026-05-01,2026-04-01,5,6,-1.081921,1.081921


In [0]:
profile_data = data[
    data[DATE_COLUMN]
    < pd.Timestamp(VALIDATION_START)
].copy()

demand_profile = (
    profile_data
    .groupby(PART_COLUMN)
    .agg(
        Average_Monthly_Usage=(
            TARGET_COLUMN,
            "mean",
        ),
        Median_Monthly_Usage=(
            TARGET_COLUMN,
            "median",
        ),
        Zero_Month_Percentage=(
            TARGET_COLUMN,
            lambda x: (x == 0).mean(),
        ),
        Nonzero_Months=(
            TARGET_COLUMN,
            lambda x: (x > 0).sum(),
        ),
        Average_Commits=(
            COMMITS_COLUMN,
            "mean",
        ),
        Median_Commits=(
            COMMITS_COLUMN,
            "median",
        ),
        Maximum_Commits=(
            COMMITS_COLUMN,
            "max",
        ),
        Months_With_Commits=(
            COMMITS_COLUMN,
            lambda x: (x > 0).sum(),
        ),
        Commit_Month_Percentage=(
            COMMITS_COLUMN,
            lambda x: (x > 0).mean(),
        ),
    )
    .reset_index()
)

demand_profile["Zero_Month_Percentage"] *= 100
demand_profile["Commit_Month_Percentage"] *= 100

for col in [
    "Average_Monthly_Usage",
    "Median_Monthly_Usage",
    "Average_Commits",
    "Median_Commits",
]:
    demand_profile[col] = demand_profile[col].round(2)

for col in [
    "Zero_Month_Percentage",
    "Commit_Month_Percentage",
]:
    demand_profile[col] = demand_profile[col].round(1)

display(demand_profile)

,fpartno,Average_Monthly_Usage,Median_Monthly_Usage,Zero_Month_Percentage,Nonzero_Months,Average_Commits,Median_Commits,Maximum_Commits,Months_With_Commits,Commit_Month_Percentage
0,058825A,641.64,568.0,0.0,36,558.33,360.61,1907.750000,36,100.0
1,060037A,1130.58,1026.5,0.0,36,673.82,455.72,2526.227723,36,100.0
2,079416,92.22,84.0,0.0,36,158.09,113.36,316.739130,36,100.0
3,080144,48.17,44.0,0.0,36,110.11,99.04,204.878049,36,100.0
4,080145,61.86,60.5,0.0,36,97.22,94.03,158.200000,36,100.0
...,...,...,...,...,...,...,...,...,...,...
439,WAS-0033,0.39,0.0,94.4,2,0.00,0.00,0.000000,0,0.0
440,WAS-0057,0.81,0.0,88.9,4,0.00,0.00,0.000000,0,0.0
441,WAS-0168,253.50,264.0,0.0,36,301.22,243.33,589.600000,36,100.0
442,WAS-0172,8.78,8.0,2.8,35,9.45,10.30,19.800000,36,100.0


In [0]:
import boto3
from sagemaker_studio import Project
import io

proj = Project()
project_s3_root = proj.s3.root

s3_parts = (
    project_s3_root
    .replace("s3://", "")
    .split("/", 1)
)

bucket = s3_parts[0]
prefix = (
    s3_parts[1]
    if len(s3_parts) > 1
    else ""
)

s3 = boto3.client("s3")

results_to_upload = results[
    [
        PART_COLUMN,
        DATE_COLUMN,
        "Training Through",
        "Predicted Usage",
        "Predicted Usage Rounded",
        "Actual Usage",
        "Error",
        "Absolute Error",
    ]
].copy()

csv_buffer = io.StringIO()

results_to_upload.to_csv(
    csv_buffer,
    index=False,
)

s3_key = (
    f"{prefix}/results/"
    "version_8_criticalparts.csv"
)

s3.put_object(
    Bucket=bucket,
    Key=s3_key,
    Body=csv_buffer.getvalue().encode(
        "utf-8"
    ),
    ContentType="text/csv",
)

s3_path = (
    f"s3://{bucket}/{s3_key}"
)

print(
    "Successfully uploaded Version 8 "
    "commitments + recency weighting results to S3!"
)

print(f"S3 path: {s3_path}")
print(f"Rows uploaded: {len(results_to_upload)}")

Successfully uploaded Version 8 commitments + recency weighting results to S3!
S3 path: s3://amazon-sagemaker-369282953854-us-east-1-7ffd8a98b34e/dzd-bokklg2avhnkh3/6vqwkr3e84m5d3/dev/results/version_8_criticalparts.csv
Rows uploaded: 2232
